Work on the parsing outputs data function.

In [1]:
main_llm_dir = '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/'

# full baseline data stored
dir_jsons = '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/'

save_fig_dir = '/Users/jnaiman/Dropbox/Paper_iConf2025/figures/'

In [28]:
from glob import glob
import pickle
import pandas as pd
import json
from copy import deepcopy
import numpy as np
import os
import matplotlib.pyplot as plt
from Levenshtein import distance as levenshtein_distance # Assuming you have python-Levenshtein installed

# debug
from importlib import reload

import seaborn as sns
import numpy as np
import re


from utils.parse_lmm_output_utils import parse_first_json

In [3]:
dirs_tmp = glob(main_llm_dir + '*')
dirs_tmp

['/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/gemini_api_2.5',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/claude_api',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/gemini_api_1.5',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/chatgpt_api']

In [4]:
dirnames = []

dn_ignore = 'gemini_2.5'

dirs = []
for d in dirs_tmp:
    dn = d.split('/')[-1].replace('_api','')
    if dn not in dn_ignore:
        dirnames.append(dn)
        dirs.append(d)

dirnames

['claude', 'gemini_1.5', 'chatgpt']

In [5]:
# find total overlap of files
files_parsed = []
for d in dirs:
    files = glob(d + '/*.pickle')
    for f in files:
        files_parsed.append(f.split('/')[-1])
files_parsed1 = list(set(files_parsed))
# now check all exist
files_parsed = []
for f in files_parsed1:
    exists = True
    for d in dirs:
        if not os.path.exists(d + '/' + f):
            exists = False
    if exists:
        files_parsed.append(f)

print("there are:", len(files_parsed), 'overlapping files so far')

there are: 200 overlapping files so far


In [42]:
# claude suggestions for fixing slashes
# Replace single backslashes with double backslashes in JSON strings
def fix_json_escapes(json_str):
    # This pattern finds backslashes that aren't already properly escaped
    return re.sub(r'(?<!\\)\\(?!["\\/bfnrt]|u[0-9a-fA-F]{4})', r'\\\\', json_str)

def fix_raw_strings(json_str):
    # Remove r" prefixes and escape backslashes in the content
    def replace_raw_string(match):
        content = match.group(1)
        # Escape backslashes
        escaped_content = content.replace('\\', '\\\\')
        return f'"{escaped_content}"'
    
    # Find r"..." patterns and replace them
    return re.sub(r'r"([^"]*)"', replace_raw_string, json_str)

# Or more comprehensive - escape all single backslashes before letters
def fix_all_latex(json_str):
    return re.sub(r'(?<!\\)\\(?=[a-zA-Z])', r'\\\\', json_str)

def fix_latex_math(json_str):
    # Add quotes around $...$ expressions
    return re.sub(r'\$([^$]+)\$', r'"\$\1\$"', json_str)

In [51]:
# def parse_json_files(dirnames, dirs, files_parsed, dir_jsons, 
#                      verbose=True):

verbose=True

if True:
    dfdict = {}
    for flag in ['image id', 'plot number', 'plot type', 'question', 
                'use list', 'model', 'model id', 'LMM Answer', 'GT Answer', 
                'Level', 'Level Type']: #, 'plot types']:
        dfdict[flag] = []
    for ifile,(dn,dr) in enumerate(zip(dirnames, dirs)):
        if verbose:
            print('')
            print('****************************************')
            print('***********', dn, '****************')
            print('****************************************')
            print('')
        for f in files_parsed:
            if verbose: print("-----------", f, '------------')

            # read in full data for some extra info
            with open(dir_jsons + f.removesuffix('.pickle')+'.json', 'r') as ff:
                j = json.load(ff)
                jfd = json.loads(j)
                #import sys; sys.exit()

            with open(dr + '/' + f,'rb') as ff:
                data,  model = pickle.load(ff)

            # loop through q/a
            for qa in data:
                # make row/file
                dfdict['image id'].append(f.removesuffix('.pickle'))
                dfdict['question'].append(qa['question'])
                dfdict['model'].append(dn)
                dfdict['model id'].append(model)
                dfdict['Level'].append(qa['Level'])
                dfdict['Level Type'].append(qa['type'])
                if 'plot number' in qa: # not figure-level
                    dfdict['plot number'].append(int(qa['plot number'].split('plot')[-1]))
                    dfdict['plot type'].append(jfd[qa['plot number']]['type'])
                else:
                    dfdict['plot type'].append(None)
                    dfdict['plot number'].append(None)
                use_list = False
                if 'choose' in qa['format']:
                    use_list = True
                elif 'please choose ' in qa['context'].lower():
                    use_list = True
                dfdict['use list'].append(use_list)
                # try loading response
                try:
                    jgt = qa['A']
                except:
                    print('no GT answer!')
                    import sys; sys.exit()
                # if string/number answer
                if type(jgt) != type({}):
                    q = qa['Q'] # {"npoints":""}
                    q = q.split('{')[-1].split('}')[0]
                    q = q.split(':')[0].replace('"','').replace("'",'')
                    #print(q)
                    #import sys; sys.exit()
                    jgt = {q:jgt}
                # llm
                raw_ans_in = qa['raw answer']
                jllm = {}
                raw_ans = raw_ans_in.replace('^', 'e') # math notation
                raw_ans = raw_ans.replace('**','e')
                raw_ans = re.sub(r'(\d*\.?\d+e)\s*\(\s*-\s*(\d+)\s*\)', r'\1-\2', raw_ans)
                raw_ans = raw_ans.replace('True', 'true').replace('False', 'false')
                if '`' not in raw_ans:
                    try:
                        jllm = json.loads(raw_ans)
                    except:
                        # try splitting
                        if '{' in raw_ans and '}' in raw_ans:
                            a = '{' + raw_ans.split('{')[-1].split('}')[0] + '}'
                            try:
                                jllm = json.loads(a)
                            except Exception as esplit:
                                if verbose: 
                                    print('[ERROR]: could not load answer for this json :', a)
                                    print('  full exception:', str(esplit))
                                #fslkfjs
                                if verbose: print('')
                                jllm[list(jgt.keys())[0]] = np.nan
                elif '```json\n' in raw_ans:
                    try:
                        a = raw_ans.split('```json\n')[-1].split('```')[0]
                        jllm = json.loads(a)
                    except Exception as e:
                        if "Expecting ',' delimiter" in str(e):
                            if verbose: 
                                print('[ERROR]: json decode error (delimiter) -- ', str(e))
                                print('   json:', a)
                            jllm[list(jgt.keys())[0]] = np.nan
                            if verbose: print('')
                        elif 'Unterminated string' in str(e):
                            if verbose: 
                                print('[ERROR]: json decode error (unterminated string) -- ', str(e))
                                print('   json:', a)
                            jllm[list(jgt.keys())[0]] = np.nan
                            if verbose: print('')
                        elif 'Expecting value:' in str(e):
                            try:
                                a = raw_ans.split('```json\n')[-1].split('```')[0]
                                a = fix_raw_strings(a)
                                jllm = json.loads(a)
                            except Exception as e2:
                                try:
                                    a = raw_ans.split('```json\n')[-1].split('```')[0]
                                    a = fix_raw_strings(a)
                                    a = fix_all_latex(a)
                                    jllm = json.loads(a)
                                except Exception as e22:
                                    try:
                                        a = raw_ans.split('```json\n')[-1].split('```')[0]
                                        a = fix_raw_strings(a)
                                        a = fix_latex_math(a)
                                        jllm = json.loads(a)
                                    except Exception as e222:
                                        if verbose: 
                                            print('[ERROR]: json decode error (expecting value, t22) -- ', str(e222))
                                            print('   json:', a)
                                        jllm[list(jgt.keys())[0]] = np.nan
                                        if verbose: print('')
                        elif 'Invalid \\escape' in str(e):
                            try:
                                a = raw_ans.split('```json\n')[-1].split('```')[0]
                                a = a.replace('\\\\', '\\')
                                jllm = json.loads(a)
                            except Exception as e2:
                                try:
                                    a = raw_ans.split('```json\n')[-1].split('```')[0]
                                    a = a.replace('\\\\', '\\')
                                    a = fix_json_escapes(a)
                                    jllm = json.loads(a)
                                except Exception as e22:
                                    if verbose: 
                                        print('[ERROR]: json decode error, t3 -- ', str(e22))
                                        print('   json:', a)
                                # if 'Invalid \\escape' in str(e2):
                                #     jllm[list(jgt.keys())[0]] = np.nan
                                # else:
                                #     fjffj
                        elif 'Extra data:' in str(e):
                            try:
                                jllm = parse_first_json(raw_ans.split('```json\n')[-1].split('```')[0])
                            except Exception as e2:
                                if verbose: 
                                    print('[ERROR]: json decode error (extra data) --', str(e2))
                                    print('   json:', a)
                                jllm[list(jgt.keys())[0]] = np.nan
                        elif 'Expecting property name enclosed in double quotes:' in str(e):
                            if verbose: 
                                print('[ERROR]: json decode error (double quotes) --', str(e))
                                print('   json:', a)
                            jllm[list(jgt.keys())[0]] = np.nan
                        elif 'Invalid control character at:' in str(e):
                            if verbose: 
                                print('[ERROR]: json decode error (invalid control char) --', str(e))
                                print('   json:', a)
                            jllm[list(jgt.keys())[0]] = np.nan
                        else:
                            print('could not load answer, 2:', raw_ans)
                            sljfsl
                else:
                    print('not sure:')
                    print(raw_ans)
                    import sys; sys.exit()

                # known issues
                if 'titles' in jllm:
                    j2 = {'title':jllm['titles']}
                    jllm = deepcopy(j2)
                if 'aspect ratio' in jllm:
                    if ':' in jllm['aspect ratio']:
                        ar = float(jllm['aspect ratio'].split(':')[0])/float(jllm['aspect ratio'].split(':')[1])
                        jllm['aspect ratio'] = ar

                # test for matching keys
                for k,v in jgt.items():
                    if k not in jllm:
                        if verbose:
                            print('[ERROR]: missing key:', k)
                            print('  question format:', qa['format'])
                            print('  GT:', jgt)
                            print('  LMM:', jllm)
                            print('  raw LMM:', raw_ans_in.split('\n')[0])
                            print('')
                        jllm[k] = np.nan
                        #import sys; sys.exit()
                    elif type(jllm[k]) != type(v):
                        if jllm[k] is None:
                            continue
                        try:
                            a = type(v)(jllm[k])
                            #print('type', type(v), 'for', jllm[k])
                            jllm[k] = a
                        except:
                            try:
                                x = np.isnan(jllm[k])
                            except:
                                try:
                                    x = jllm[k].split(' ')[-1]
                                    a = type(v)(x)
                                    jllm[k] = a
                                except:
                                    # try for floats
                                    try:
                                        if type(v) == type(1.0) and type(jllm[k]) == type('string'):
                                            if '/' in jllm[k]:
                                                tofloat = float(jllm[k].split('/')[0])*1.0/float(jllm[k].split('/')[-1])
                                            else:
                                                tofloat = None
                                        else:
                                            tofloat = None
                                        if tofloat is None:
                                            print('[ERROR]: different types of values, could not fix:')
                                            print('  GT:', v, type(v))
                                            print('  LLM:', jllm[k], type(jllm[k]))
                                            print('  raw LLM:', raw_ans_in.split('\n')[0])
                                            print('')
                                        jllm[k] = tofloat
                                    except:
                                        if verbose:
                                            print('[ERROR]: different types of values:')
                                            print('  GT:', v, type(v))
                                            print('  LLM:', jllm[k], type(jllm[k]))
                                            print('  raw LLM:', raw_ans_in.split('\n')[0])
                                            print('')
                                        if type(jllm[k]) == type(''):
                                            jllm[k] = None
                                        else:
                                            laksjl
                            #import sys; sys.exit()
                # drop non-presents
                jllm_tmp = deepcopy(jllm)
                for k,v in jllm_tmp.items():
                    if k not in jgt:
                        del jllm[k]

                dfdict['LMM Answer'].append(jllm)
                dfdict['GT Answer'].append(jgt)
                    
                
    df = pd.DataFrame(dfdict)
    #return df


****************************************
*********** claude ****************
****************************************

----------- Picture153.pickle ------------
----------- Picture154.pickle ------------
----------- Picture35.pickle ------------
----------- Picture176.pickle ------------
----------- Picture175.pickle ------------
[ERROR]: json decode error (unterminated string) --  Unterminated string starting at: line 12 column 82 (char 766)
   json: {
  "ytick values": [
    [10, 100],
    [-20, -18, -16, -14, -12, -10, -8, -6],
    [30, 40, 50, 60, 70, 80, 90, 100, 110],
    [0, 50, 100, 150, 200],
    [-8000, -6000, -4000, -2000, 0, 2000, 4000, 6000],
    [-2000, 0, 2000, 4000, 6000, 8000],
    [-60, -40, -20, 0, 20, 40, 60, 80, 100],
    ["$10e{-14}$", "$10e{-12}$", "$10e{-10}$", "$10e{-8}$", "$10e{-6}$", "$10e{-4}$", "$10e{-2}$", "$10e{0}$", "$10e{2}$", "$10e{4}$", "$10e{6}$", "$10e{8}$", "$10e{10}$", "$10e{12}$", "$10e{14}$"],
    ["$10e{-7}$", "$10e{-6}$", "$10e{-5}$", "$10e{

In [52]:
# a = '{"x-axis errors": True}'
# json.loads(a)

In [21]:
a

'{"ytick values": [[-2, -1, 0], [0, 20, 40], [0, 100, 200], [10^-2, 10^-1, 10^0, 10^1, 10^2, 10^3], [0, 1, 2], [""], [0, 50, 100], [-25, 0, 25]]}'

In [23]:
a[61:]

'0^-2, 10^-1, 10^0, 10^1, 10^2, 10^3], [0, 1, 2], [""], [0, 50, 100], [-25, 0, 25]]}'

In [11]:
# df = parse_json_files(dirnames, dirs, files_parsed, dir_jsons, 
#                      verbose=True)